### Pandapower with UK Power Networks

This tutorial shows some functionalities and studies that can be performed using the power flow capabilities of pandapower. 
It will demonstrate how to run power flow simulations in pandapower, how to perform grid analyses and investigate different use cases relying on the power flow engine of pandapower.

This tutorial has been created in collaboration with UK Power Networks, the Distribution System Operator owning and operating the electricity network across London, the South East and the East of England.

The tutorial will use the real grids associated with the three licensed electricity distribution networks operated by UK Power Networks (LPN, SPN and EPN).
It will provide some examples of how pandapower can be used to run investigations and analyses using the open source data released by UK Power Networks.

UK Power Networks has provided the grid data as part of their LTDS CIM dataset release. It is a "Shared" dataset that requires special access. To request access, visit the [LTDS CIM](https://ukpowernetworks.opendatasoft.com/explore/dataset/ukpn-ltds-cim/information/) page and complete the [Shared Data Request Form](https://ukpowernetworks.opendatasoft.com/login/?next=/explore/forms/cim-access-request-form/). Once approved, CIM data is published as XML file attachments (one per licence area: EPN, SPN, LPN). You can download the XML files directly from the portal.

The additional data required to integrate load and generation in the grid are openly available as Excel tables at the following links: 
- EPN --> [EPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FEPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- SPN --> [SPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FSPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)
- LPN --> [LPN Long Term Development Statement - November 2025](https://ukpowernetworks.sharepoint.com/sites/OpenDataPortalLibrary/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FOpenDataPortalLibrary%2FShared%20Documents%2FGeneral%2FLong%20Term%20Development%20Statement%2FNovember%202025%2FLPN%20Long%20Term%20Development%20Statement%20%2D%20November%202025&p=true&ga=1)


In [ ]:
# Import the needed libraries 
import pandapower as pp
import pandas as pd
import numpy as np
import os
pd.options.display.float_format = '{:,.4f}'.format

#### Import of the UK Power Network grids
This tutorial assumes that the grids of UK Power Networks have been already imported from the CIM data and saved as pandapower networks in json format. 
To see how to import the UK Power Networks grids starting from the CIM files downloadable from the UK Power Networks portal, please refer to the following [UKPN_CIM2pp_tutorial](). 
Here you can also find how to save the pandapower grid into a json file and how to navigate through the pandapower grid data or the attributes of the different grid components. 

In [ ]:
# Import the grid for the analysis
filename = "LPN EQ SSH_0401_eq.json"   # Give here the name of the json file with the UKPN grid you want to use
if os.path.isfile(filename):
    net = pp.from_json(filename)
else:
    print("file does not exist, creating a dummy net")
    net = pp.create_empty_network()
    bus = pp.create_bus(net, vn_kv=132)
    pp.create_ext_grid(net, bus=bus)

#### Workarounds for power flow execution
The following blocks of code provide some functions to apply some workarounds necessary to run successfully the power flow on the UK Power Networks grids.
These workarounds include, for example, the creation of external grids (*slack buses* in the power flow terminology) or the replacement of zero impedance components with switches. 

In [ ]:
# Function to replace components with very small impedance with switches.
from pandapower.toolbox import create_replacement_switch_for_branch

def _replace_zero_impedance_components(net):
    min_ohm = 0.001
    to_replace = (np.abs(net.line.x_ohm_per_km * net.line.length_km) <= min_ohm) & net.line.in_service

    if np.any(to_replace):
        print(f"replaced {sum(to_replace)} lines with switches")

    for i in net.line.loc[to_replace].index.values:
        create_replacement_switch_for_branch(net, "line", i)
        net.line.at[i, "in_service"] = False

    xward = net.xward.loc[(np.abs(net.xward.x_ohm) <= min_ohm) & net.xward.in_service].index.values
    if len(xward) > 0:
        pp.replace_xward_by_ward(net, index=xward, drop=False)
        print(f"replaced {len(xward)} xwards with wards")

    zb_f_ohm = np.square(net.bus.loc[net.impedance.from_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    zb_t_ohm = np.square(net.bus.loc[net.impedance.to_bus.values, "vn_kv"].values) / net.impedance.sn_mva
    impedance = ((np.abs(net.impedance.xft_pu) <= min_ohm / zb_f_ohm) |
                (np.abs(net.impedance.xtf_pu) <= min_ohm / zb_t_ohm)) & net.impedance.in_service

    if any(impedance):
        print(f"replaced {sum(impedance)} impedance elements with switches")

    for i in net.impedance.loc[impedance].index.values:
        pp.create_replacement_switch_for_branch(net, "impedance", i)
        net.impedance.at[i, "in_service"] = False

In [ ]:
# Function to apply the needed workarounds
def apply_workarounds(net, license_area):
    net.impedance.drop(net.impedance.index, inplace=True)
    _replace_zero_impedance_components(net)
    net.line["c_nf_per_km"] *= 0.1
    net.load["p_mw"] *= 0.1

    if license_area == "LPN":
        pp.create_ext_grid(net,bus=10711,vm_pu=1)
        pp.create_ext_grid(net,bus=10699,vm_pu=1)
        pp.create_ext_grid(net,bus=10674,vm_pu=1)
        pp.create_ext_grid(net,bus=10738,vm_pu=1)
        pp.create_ext_grid(net,bus=10673,vm_pu=1)
    elif license_area == "SPN":
        pp.create_ext_grid(net,bus=4899,vm_pu=1)
        pp.create_ext_grid(net,bus=4879,vm_pu=1)
        pp.create_ext_grid(net,bus=4903,vm_pu=1)
        pp.create_ext_grid(net,bus=4920,vm_pu=1)
        pp.create_ext_grid(net,bus=4916,vm_pu=1)
        pp.create_ext_grid(net,bus=4925,vm_pu=1)
        pp.create_ext_grid(net,bus=4878,vm_pu=1)
    elif license_area == "EPN":
        pp.create_ext_grid(net,bus=9906,vm_pu=1)
        pp.create_ext_grid(net,bus=9918,vm_pu=1)
        pp.create_ext_grid(net,bus=9900,vm_pu=1)
        pp.create_ext_grid(net,bus=9910,vm_pu=1)
        pp.create_ext_grid(net,bus=9878,vm_pu=1)
    else:
        raise ValueError("Sorry, this license area does not exist in UK Power Networks. Allowed areas are LPN, SPN and EPN.")

    return net


In [ ]:
# Apply the workarounds on the selected grid
license_area = "LPN"  # Provide here the name of the considered license area. It should be "LPN", "SPN", or "EPN".
if net.bus.index.size > 1:
    net = apply_workarounds(net, license_area)

### Contingency analysis: run a power flow study
One of the easiest tasks that can be done with pandapower is to run a power flow. 
This allows analysing the voltage conditions in the grid and the powers/currents flowing through the different lines and components of the network, considering the load and generation available as input.

Through a power flow calculation it is possible to make a contingency analysis, namely to assess if the operating conditions of the grid are within the allowed boundaries.

In this section, you will see: 
- How to run a power flow and visualize the results
- How to filter the power flow results
- How to identify possible contingencies (overloading or voltage violations)



In [ ]:
# Run a power flow
pp.runpp(net, max_iteration=50)

In the bus results table you will find the resulting bus voltage and power consumption / injection at each bus

In [ ]:
# Visualize bus results
display(net.res_bus)
display("Maximum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)))
display("Minimum voltage magnitude in the grid (per unit): " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)))

Some of the bus results may have NaN. This happens for those buses that are disconnected from the main grid.

In [ ]:
# Visualize number of connected buses
num_disconnected_buses = np.sum(np.isnan(net.res_bus.vm_pu))
num_connected_buses = np.sum(~np.isnan(net.res_bus.vm_pu))
total_num_buses = len(net.bus)
percentage_connected_buses = 100 * num_connected_buses / total_num_buses
display("Percentage of connected buses: " + "{:.2f}".format(percentage_connected_buses) + "%")

In the line and transformer result tables you can see, among others, the level of power flowing through these components.

In [ ]:
# Visualize line results
display(net.res_line)
display("Maximum active power in the lines: " + "{:.2f}".format(net.res_line.loc[net.res_line.p_from_mw.notna(), 'p_from_mw'].max()) + " MW")

In [ ]:
# Visualize transformer results
display(net.res_trafo)
display("Maximum active power in the transformers: " + "{:.2f}".format(net.res_trafo.loc[net.res_trafo.p_hv_mw.notna(), 'p_hv_mw'].max()) + " MW")

You can easily sort the results using the *sort_values* function


In [ ]:
# Sort bus results from buses with the smallest voltage
net.res_bus.sort_values("vm_pu").head(20)

In [ ]:
# Sort line results from lines with highest active power flow
net.res_line.sort_values("p_from_mw", ascending=False).head(20)

You can visualize the results for a specific element

In [ ]:
# Visualize bus results at bus 45
bus_idx = 45
if bus_idx in net.res_bus.index:
    print(net.res_bus.loc[bus_idx])
else:
    print("The given bus does not exist")

In [ ]:
# Visualize results for transformer 15
if 15 in net.res_trafo.index:
    print(net.res_trafo.loc[15])
else:
    print("The given transformer does not exist")

You can filter the results as you like, selecting only specific types or clusters of elements, or specific columns of the tables

In [ ]:
# Visualize bus results only for buses at 132 kV
net.res_bus[net.bus.vn_kv==132]

In [ ]:
# Visualize transformer results only for 132 kV/33 kV  transformers 
net.res_trafo[(net.trafo.vn_hv_kv==132) & (net.trafo.vn_lv_kv==33)]

In [ ]:
# Visualize bus results only for a desired zone (zones can be seen at net.bus.zone)
if 'zone' not in net.bus:
    net.bus['zone'] = ''
if license_area == "LPN":
    zonename = "Fulham Palace Rd C"
elif license_area == "SPN":
    zonename = "South Hove"
elif license_area == "EPN":
    zonename = "Norwich Main"

net.res_bus[net.bus.zone==zonename]

In [ ]:
# Visualize only active and reactive powers of the lines
net.res_line[["p_from_mw", "q_from_mvar", "p_to_mw", "q_to_mvar"]]

You can easily identify possible voltage contingencies in the grid, namely voltage values beyond the allowed thresholds. 

In [ ]:
# Check possible voltage violations
# Define voltage boundaries
lower_v_threshold = 0.90   # Define the lower boundary of the voltage magnitude (in per unit)
upper_v_threshold = 1.10   # Define the upper boundary of the voltage magnitude (in per unit)

# Check for overvoltages
if np.any(net.res_bus.vm_pu > upper_v_threshold):
    display("Overvoltages are present in the grid. Maximum voltage is: " + "{:.4f}".format(np.nanmax(net.res_bus.vm_pu)) + " p.u.")
    buses_with_overvoltage = net.bus.index[net.res_bus.vm_pu>upper_v_threshold]
else: 
    display("No overvoltages are present in the grid")

# Check for undervoltages
if np.any(net.res_bus.vm_pu < lower_v_threshold):
    display("Undervoltages are present in the grid. Minimum voltage is: " + "{:.4f}".format(np.nanmin(net.res_bus.vm_pu)) + " p.u.")
    buses_with_undervoltage = net.bus.index[net.res_bus.vm_pu<lower_v_threshold]
else: 
    display("No undervoltages are present in the grid")



You can easily check if any overloading exists in the grid (**NB**: the possibility of verifying overloadings depends on the availability of rated values for lines and/or transformers).

In [ ]:
# Check overloadings for transformers

overloading_factor = 1  # You can define an overloading factor if you desire to check overloadings for values different from 100%
if np.any(net.res_trafo.loading_percent > 100*overloading_factor):
    display("Overloading present in the grid transformers. Maximum loading is: " + "{:.2f}".format(np.nanmax(net.res_trafo.loading_percent)) + "%")
else: 
    display("No overloading is present in the grid transformers")

In [ ]:
# Visualize transformer loading (results sorted by the largest loading)
net.res_trafo[["loading_percent"]].sort_values("loading_percent", ascending=False)

### Grid analysis and forecasting: impact of different operating conditions

Pandapower allows easily modifying the default data to test different loading or generation levels. This is for example useful to analyse future scenarios or to perform grid analyses with forecasted values.

In this section you will see: 
- How to change load and generation values
- How to scale up or down specific categories of loads or generation

A load or generation value, if desired, can be simply overwritten.

In [ ]:
# Change active power at load (Note: the index of the load is not the same as the index of the bus to which it is connected)
load_index = 7          # index of the load to be overwritten
new_load_p = 0.87       # value of the active power in MW
net.load.loc[load_index, "p_mw"] = new_load_p
if np.isnan(net.load.loc[load_index, 'bus']):
    net.load.loc[load_index, ["bus", "q_mvar", "in_service", "scaling"]] = [0, 0, True, 1]
    net.load.bus = net.load.bus.astype(int)
    net.load.in_service = net.load.in_service.astype(bool)
net.load.loc[load_index]

In [ ]:
# Change active and reactive power at static generator
if license_area == "LPN":
    sgen_index = 7
elif license_area == "SPN":
    sgen_index = 0
elif license_area == "EPN":
    sgen_index = 4        # index of the sgen to be overwritten
new_sgen_p = 1.2        # value of the active power in MW
new_sgen_q = 0.2        # value of the reactive power in Mvar
net.sgen.loc[sgen_index, "p_mw"] = new_sgen_p
net.sgen.loc[sgen_index, "q_mvar"] = new_sgen_q
if np.isnan(net.sgen.loc[sgen_index, 'bus']):
    net.sgen.loc[sgen_index, ["bus", "in_service", "scaling"]] = [0, True, 1]
    net.sgen.bus = net.sgen.bus.astype(int)
    net.sgen.in_service = net.sgen.in_service.astype(bool)
net.sgen.bus = net.sgen.bus.astype(int)
net.sgen.loc[sgen_index]

In [ ]:
# Run power flow with the new data
pp.runpp(net, max_iteration=50)

The resulting power at the bus with the modified load and sgen will now correspond to the modified values given in input

In [ ]:
# Visualize results at the buses with modified load
net.res_bus.loc[net.load.loc[7, "bus"]]

In [ ]:
# Visualize results at the buses with modified sgen
net.res_bus.loc[net.sgen.loc[sgen_index, "bus"]]

It is possible also to scale up or down all loads/sgens, or a subset of them, using the *scaling* attribute available for both loads and static generators.

In [ ]:
# Scale all loads
net.load.scaling = 0.5   # this will scale down all loads to 50% of the power available in the p_mw and q_mvar fields.
net.load

In [ ]:
# Run power flow with the new data
pp.runpp(net, max_iteration=50)
# Visualize results at the buses with modified load --> NOTE: p_mw result will be scaled according to scaling factor used
net.res_bus.loc[net.load.loc[load_index, "bus"]]

Scale values only for a specific zone:

In [ ]:
# Find to which zone each load belongs to
load_zone = net.bus.zone[net.load.bus]
# Apply a scaling factor only for the desired zone
if license_area == "LPN":
    zonename="Fulham Palace Rd C"
elif license_area == "SPN":
    zonename="South Hove"
elif license_area == "EPN":
    zonename="Harlow West Grid"
net.load.loc[(load_zone==zonename).values, "scaling"] = 0.7
net.load.loc[(load_zone==zonename).values]


If loads and generators are classified with different *types*, it is possible to apply different scaling factors for each *type*. This is for example useful to apply different scaling factor for different generation technologies (e.g., PV, wind, etc.) and to modify the load and generation for the different clusters at different time steps, during a time series simulation. 

In [ ]:
# Visualize the generator type
net.sgen.type

In [ ]:
# Change the scaling factor for a specific type of generation
net.sgen.loc[net.sgen.type=="PV", "scaling"] = 0.2

### Hosting capacity: impact of new load or generation connections

Hosting capacity studies are a common use case that can be addressed leveraging the pandapower power flow libraries. The goal is to understand how much load or generation can be connected to a bus, before exceeding the allowed boundaries (voltage boundaries or overloading of the grid components).

In this section you will see:
- How to add new loads or generators to the grid
- How to discover the maximum load or generation that can be added at a bus before exceeding the operational boundaries (i.e., voltage or overloading limits)

A new load can be easily created with the *create_load" function of pandapower. It requires defining the bus to which the load will be connected and its active and reactive power.

In [ ]:
# Create a new load at the desired bus
load_bus = 3403

if load_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=load_bus)
load_p = 0.2
load_q = 0.1
pp.create_load(net, bus=load_bus, p_mw=load_p, q_mvar=load_q)
net.load.tail(1)

A new static generator can be easily created with the *create_sgen" function of pandapower. It requires defining the bus to which the static generator will be connected and its active and reactive power.

In [ ]:
# Create a new sgen at the desired bus
sgen_bus = 3403

if sgen_bus not in net.bus.index:
    # if the bus not exists, it needs to be created first
    pp.create_bus(net, vn_kv=132, index=sgen_bus)
sgen_p = 0.3
sgen_q = 0
pp.create_sgen(net, bus=sgen_bus, p_mw=sgen_p, q_mvar=sgen_q)
net.sgen.tail(1)

It is possible to run a hosting capacity study and understand how much load or generation can be connected to a particular node, by incrementing continuously the power (of the load or generator) till when the boundaries of interest are not exceeded.

In this example, for simplicity, we will investigate how much load can be added to the desired bus before exceeding the loading capacity of the grid transformers.

In [ ]:
if license_area == "LPN":
    hosting_bus = 8
elif license_area == "SPN":
    hosting_bus = 36
elif license_area == "EPN":
    hosting_bus = 20
else:
    hosting_bus = 0             # bus selected for the analysis
            # bus selected for the analysis
incremental_p_mw = 1        # incremental value of power
if hosting_bus not in net.bus.index:
    print("The given bus does not exist")
    hosting_bus = net.bus.index[0]   # replace the bus with the first bus in the grid
load_index = pp.create_load(net, bus=hosting_bus, p_mw=0, q_mvar=0)
within_hosting_limit = True    # boolean telling if we are still within inside the allowed boundary

# Hosting capacity logic
while within_hosting_limit:
    net.load.loc[load_index, "p_mw"] += incremental_p_mw
    pp.runpp(net, max_iteration=50)
    if np.any(net.res_trafo.loading_percent > 100) or net.trafo.index.size == 0:
        within_hosting_limit = False
        net.load.loc[load_index, "p_mw"] -= incremental_p_mw

# Visualize maximum load that can be connected at the selected bus
display("Maximum load that can be connected at bus " + str(load_index) + " is " + str(net.load.loc[load_index, "p_mw"]) + " MW")

### Grid control: impact of different settings for controllable components 

The operating conditions of the grid can be changed in multiple ways, acting on controllable components. Pandapower allows manipulating controllable components (like switches, capacitor banks, transformers with tap changers, etc.) to test the impact of different settings.

In this section you will see:
- How to change status of switches and evaluate the impact of different network topologies
- How to change tap position of transformers and assess the resulting impact
- How to connect or disconnect capacitor banks and assess the resulting impact

**Switches** are among the components that can controlled to modify how the power flows through the grid, as their open or closed status will determine the final topology of the grid. In pandapower, the status of the switch can be modified simply by acting on its *closed* attribute

In [ ]:
# Visualize the attributes of a switch
switch_index = 0
if switch_index in net.switch.index:
    print(net.switch.loc[switch_index])
else:
    print("The given switch does not exist")

In [ ]:
# Visualize if the switch is closed (closed attribute = True) or open (closed attribute = False)
if switch_index not in net.switch.index:
    bus_idx_1 = pp.create_bus(net, vn_kv=132)
    bus_idx_2 = pp.create_bus(net, vn_kv=132)
    pp.create_switch(net, bus_idx_1, bus_idx_2, 'b', True, index=switch_index)
net.switch.loc[switch_index, "closed"]

In [ ]:
# Change the status of a switch
net.switch.loc[switch_index, "closed"] = False   # In this case, we are opening the switch
net.switch.loc[switch_index]

The **tap position of transformers** is another parameter that can be modified to affect the operating conditions of the grid. In particular, through the transformer tap position it is possible to modify the resulting voltage levels.  

In [ ]:
# Visualize the attributes of a transformer
trafo_index = 0
if trafo_index in net.trafo.index:
    print(net.trafo.loc[trafo_index])
else:
    print("The given trafo does not exist")

In [ ]:
# Visualize the main tap changer settings of a transformer
if trafo_index not in net.trafo.index:
    bus1 = pp.create_bus(net, vn_kv=110, name="Bus 110kV-1")
    bus2 = pp.create_bus(net, vn_kv=20, name="Bus 20kV-2")
    trafo = pp.create_transformer(net, hv_bus=bus1, lv_bus=bus2, std_type="63 MVA 110/20 kV", index=trafo_index)
net.trafo.loc[trafo_index, ["tap_min", "tap_max", "tap_neutral", "tap_pos"]]

In [ ]:
# Visualize voltage at the transformer secondary bus before applying any change
pp.runpp(net, max_iteration=50)
display("Voltage at the low voltage side of the transformer: " + "{:.4f}".format(net.res_bus.loc[net.trafo.loc[trafo_index,"lv_bus"], "vm_pu"]) + " p.u.")

In [ ]:
# Change the tap position of the selected transformer
net.trafo.loc[trafo_index, "tap_pos"] = 2   # In this case, we are forcing the transformer to have tap position 2
net.trafo.loc[trafo_index]

In [ ]:
# Visualize voltage at the transformer secondary bus after applying the tap position change
pp.runpp(net, max_iteration=50)
display("Voltage at the low voltage side of the transformer: " + "{:.4f}".format(net.res_bus.loc[net.trafo.loc[trafo_index,"lv_bus"], "vm_pu"]) + " p.u.")

**Capacitor banks** (or, more in general, shunts) can also affect the operating conditions by bringing an injection of reactive power in the grid. In pandapower, it is possible to connect or disconnect shunts by acting on their "in_service" attribute

In [ ]:
# Visualize the attributes of a shunt
shunt_index = 3
if shunt_index in net.shunt.index:
    print(net.shunt.loc[shunt_index])
else:
    print("The given shunt does not exist")

In [ ]:
# Visualize if the shunt is connected (in_service = True) or not (in_service = False)
if shunt_index not in net.shunt.index:
    bus = pp.create_bus(net, vn_kv=110, name="Bus 110kV-1")
    pp.create_shunt(net, bus=bus, q_mvar=0.5, p_mw=0.0, index=shunt_index)
net.shunt.loc[shunt_index, "in_service"]

In [ ]:
# Visualize voltage at the shunt bus before applying any change
pp.runpp(net, max_iteration=50)
display("Voltage at the shunt bus: " + "{:.4f}".format(net.res_bus.loc[net.shunt.loc[shunt_index,"bus"], "vm_pu"]) + " p.u.")


In [ ]:
# Change the status of the switch
if net.shunt.loc[shunt_index, "in_service"]:
    net.shunt.loc[shunt_index, "in_service"] = False
else:
    net.shunt.loc[shunt_index, "in_service"] = True

net.shunt.loc[shunt_index]

In [ ]:
# Visualize voltage at the shunt bus after applying the change
pp.runpp(net, max_iteration=50)
display("Voltage at the shunt bus: " + "{:.4f}".format(net.res_bus.loc[net.shunt.loc[shunt_index,"bus"], "vm_pu"]) + " p.u.")